# AutoStitch — IDM-VTON LoRA Fine-Tuning (v2, consolidated)

Run cells in order, top to bottom. This version includes every fix found while debugging: pinned dependency versions, the flax/jax import workaround, dataset symlinks, the JSON path fix, and a custom dataset class that derives the inpaint mask from the `agnostic-v3.2` images (since this Kaggle copy of VITON-HD doesn't ship a separate mask file).

**Before running:** make sure Runtime → Change runtime type → GPU (T4) is selected.

## Cell 1: Install dependencies (pinned, compatible versions)

In [ ]:
!pip install -q diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 gradio==3.50.2 torchvision fvcore einops omegaconf python-multipart fastapi uvicorn
!pip install -q peft==0.7.1 bitsandbytes==0.41.3
!pip uninstall -y jax jaxlib flax -q

## Cell 2: Disable flax/tf checks in transformers
Must run **before** any `transformers` import, in every fresh session.

In [ ]:
import os
os.environ["USE_FLAX"] = "0"
os.environ["USE_TF"] = "0"

## Cell 3: Clone IDM-VTON

In [ ]:
%cd /content
if not os.path.exists('IDM-VTON'):
    !git clone https://github.com/yisol/IDM-VTON.git
%cd /content/IDM-VTON

## Cell 4: Confirm GPU

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > GPU before continuing.")

## Cell 5: Download VITON-HD dataset from Kaggle
Requires a Kaggle API token. Get one from kaggle.com → Settings → API → Generate New Token, then paste it below.

In [ ]:
KAGGLE_TOKEN = "PASTE_YOUR_KAGGLE_API_TOKEN_HERE"

!mkdir -p ~/.kaggle && echo "$KAGGLE_TOKEN" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!pip install -q kaggle
!kaggle datasets download -d marquis03/high-resolution-viton-zalando-dataset -p /content/vitonhd_raw --unzip

## Cell 6: Link dataset folders into IDM-VTON/train and fix path mismatches
- Symlinks each VITON-HD subfolder into `IDM-VTON/train/`
- Copies `vitonhd_train_tagged.json` into `train/` (repo ships it at root)
- Links `agnostic-v3.2` as `agnostic-mask` for naming consistency

In [ ]:
import os, shutil

os.makedirs('/content/IDM-VTON/train', exist_ok=True)

folders_to_link = ['image', 'cloth', 'cloth-mask', 'image-densepose',
                    'agnostic-v3.2', 'image-parse-v3',
                    'image-parse-agnostic-v3.2', 'openpose_img', 'openpose_json']

for folder in folders_to_link:
    src = f'/content/vitonhd_raw/train/{folder}'
    dst = f'/content/IDM-VTON/train/{folder}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"Linked {folder}")
    elif os.path.exists(dst):
        print(f"{folder} already linked")
    else:
        print(f"MISSING source: {src}")

if os.path.exists('/content/vitonhd_raw/train_pairs.txt'):
    shutil.copy('/content/vitonhd_raw/train_pairs.txt', '/content/IDM-VTON/train/train_pairs.txt')
    print("train_pairs.txt copied")

# JSON: repo ships vitonhd_train_tagged.json at IDM-VTON root; dataset class expects it in train/
root_json = '/content/IDM-VTON/vitonhd_train_tagged.json'
train_json = '/content/IDM-VTON/train/vitonhd_train_tagged.json'
if os.path.exists(root_json) and not os.path.exists(train_json):
    shutil.copy(root_json, train_json)
    print("Copied json into train/")

# agnostic-mask naming fix
if not os.path.exists('/content/IDM-VTON/train/agnostic-mask'):
    os.symlink('/content/vitonhd_raw/train/agnostic-v3.2', '/content/IDM-VTON/train/agnostic-mask')
    print("Linked agnostic-v3.2 as agnostic-mask")

print("\nContents of train/:")
!ls /content/IDM-VTON/train/

## Cell 7: Dataset class (mask derived from agnostic-v3.2 gray region)
Adapted from IDM-VTON's own `train_xl.py` `VitonHDDataset`. The only
change: instead of loading a precomputed `agnostic-mask` PNG (which
this Kaggle copy doesn't provide), it derives the inpaint mask by
detecting the solid mid-gray (~128,128,128) erased region in the
`agnostic-v3.2` RGB image.

In [ ]:
import random
import json
import numpy as np
from torchvision import transforms
from PIL import Image
from transformers import CLIPImageProcessor
from typing import Literal, Tuple
import torch.utils.data as data

class VitonHDDatasetFixed(data.Dataset):
    def __init__(self, dataroot_path, phase: Literal["train", "test"],
                 order: Literal["paired", "unpaired"] = "paired",
                 size: Tuple[int, int] = (512, 384)):
        super().__init__()
        self.dataroot = dataroot_path
        self.phase = phase
        self.height, self.width = size
        self.norm = transforms.Normalize([0.5], [0.5])
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        self.toTensor = transforms.ToTensor()

        with open(os.path.join(dataroot_path, phase, f"vitonhd_{phase}_tagged.json"), "r") as f:
            data1 = json.load(f)

        annotation_list = ["sleeveLength", "neckLine", "item"]
        self.annotation_pair = {}
        for k, v in data1.items():
            for elem in v:
                annotation_str = ""
                for template in annotation_list:
                    for tag in elem["tag_info"]:
                        if tag["tag_name"] == template and tag["tag_category"] is not None:
                            annotation_str += tag["tag_category"] + " "
                self.annotation_pair[elem["file_name"]] = annotation_str

        self.order = order
        im_names, c_names = [], []
        filename = os.path.join(dataroot_path, f"{phase}_pairs.txt")
        with open(filename, "r") as f:
            for line in f.readlines():
                if phase == "train":
                    im_name, _ = line.strip().split()
                    c_name = im_name
                else:
                    if order == "paired":
                        im_name, _ = line.strip().split()
                        c_name = im_name
                    else:
                        im_name, c_name = line.strip().split()
                im_names.append(im_name)
                c_names.append(c_name)

        self.im_names = im_names
        self.c_names = c_names
        self.flip_transform = transforms.RandomHorizontalFlip(p=1)
        self.clip_processor = CLIPImageProcessor()

    def _derive_mask_from_agnostic(self, im_name):
        agnostic_path = os.path.join(self.dataroot, self.phase, "agnostic-v3.2", im_name)
        agnostic = Image.open(agnostic_path).convert("RGB").resize((self.width, self.height))
        arr = np.array(agnostic).astype(np.int16)
        gray_dist = np.abs(arr - 128).sum(axis=-1)
        mask_arr = (gray_dist < 20).astype(np.float32)
        mask = torch.from_numpy(mask_arr).unsqueeze(0)
        return mask

    def __getitem__(self, index):
        c_name = self.c_names[index]
        im_name = self.im_names[index]
        cloth_annotation = self.annotation_pair.get(c_name, "shirts")

        cloth = Image.open(os.path.join(self.dataroot, self.phase, "cloth", c_name))
        im_pil_big = Image.open(os.path.join(self.dataroot, self.phase, "image", im_name)).resize((self.width, self.height))
        image = self.transform(im_pil_big)

        mask = self._derive_mask_from_agnostic(im_name)

        densepose_map = Image.open(os.path.join(self.dataroot, self.phase, "image-densepose", im_name))
        pose_img = self.toTensor(densepose_map)

        if self.phase == "train" and random.random() > 0.5:
            cloth = self.flip_transform(cloth)
            mask = self.flip_transform(mask)
            image = self.flip_transform(image)
            pose_img = self.flip_transform(pose_img)

        cloth_trim = self.clip_processor(images=cloth, return_tensors="pt").pixel_values
        im_mask = image * (1 - mask)
        pose_img = self.norm(pose_img)

        return {
            "c_name": c_name,
            "image": image,
            "cloth": cloth_trim,
            "cloth_pure": self.transform(cloth),
            "inpaint_mask": mask,
            "im_mask": im_mask,
            "caption": "model is wearing " + cloth_annotation,
            "caption_cloth": "a photo of " + cloth_annotation,
            "pose_img": pose_img,
        }

    def __len__(self):
        return len(self.im_names)

## Cell 8: Test the dataset class on a single sample

In [ ]:
test_dataset = VitonHDDatasetFixed(dataroot_path="/content/IDM-VTON/train", phase="train", size=(512, 384))
print("Dataset size:", len(test_dataset))
sample = test_dataset[0]
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(k, v.shape)
    else:
        print(k, v)

## What's next (not yet in this notebook)

Once Cell 8 runs cleanly, the remaining work is: loading IDM-VTON's actual
UNet + garment-encoder UNet (from `train_xl.py`'s `main()`), wrapping the
UNet with a `peft` LoRA config, building a `DataLoader` around this dataset
class (ideally on a small subset first, given T4's 15GB VRAM), and running
a training loop adapted from `train_xl.py`'s core loop. That's a heavier,
riskier step — bring back whatever error or output Cell 8 gives, and we'll
build that part next together rather than guessing it all at once.